NAME : VIKAS MALVIYA

SCH. NO. : 25215011109

SUBJECT : LAB 05

TITLE:  LLM- prompt engineering



In [ ]:
# Install required libraries (run once)
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# Check GPU availability
print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA Available: True
Device: Tesla T4


In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
def generate_response(prompt, max_tokens=150):
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
zero_shot_prompt = "Explain Graph Neural Networks in one paragraph."

zero_shot_output = generate_response(zero_shot_prompt)

print("----- Zero Shot Output -----")
print(zero_shot_output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


----- Zero Shot Output -----
[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of deep learning model specifically designed to analyze and learn from graph data structures, which consist of nodes representing entities and edges representing relationships between them. Unlike traditional neural networks that process data in a flat, Euclidean space, GNNs use messages passed between neighboring nodes and their aggregation functions to learn node representations that capture local structural information. Through multiple layers of message passing and non-linear transformations, GNNs can effectively capture both local and global graph structures, making them powerful tools for various graph-based applications such as social network analysis, recommendation systems, and molecular chemistry.


In [ ]:
few_shot_prompt = """
Translate the following English sentences to French.

Example 1:
English: I love machine learning.
French: J'aime l'apprentissage automatique.

Example 2:
English: This course is very interesting.
French: Ce cours est très intéressant.

Now translate:
English: Artificial Intelligence is transforming the world.
"""

few_shot_output = generate_response(few_shot_prompt)

print("----- Few Shot Output -----")
print(few_shot_output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


----- Few Shot Output -----
[INST] 
Translate the following English sentences to French.

Example 1:
English: I love machine learning.
French: J'aime l'apprentissage automatique.

Example 2:
English: This course is very interesting.
French: Ce cours est très intéressant.

Now translate:
English: Artificial Intelligence is transforming the world.
 [/INST] French: L'Intelligence Artificielle est transformant le monde.
